<a href="https://colab.research.google.com/github/shahad030/Final-Project--Bayan-bilingual-citizen-feedback/blob/main/Copy_of_Day4_Bayan_Lab_with_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 — اجعل Bayan أسرع وشغّله كخدمة
**SDA-AIE-211 **

## الهدف
سنقوم بأربع خطوات فقط:
1. نقيس سرعة نموذج التصنيف.
2. نحوله إلى ONNX ثم INT8 ونقارن السرعة والنتائج.
3. نشغّل النموذج داخل API.
4. نختبر الخدمة ونوثّق النتائج.

**قبل البدء:** أكمل Lab 3a، وتأكد من وجود:
- `bayan_topic_model`
- `data/bench_mix.npy`

> شغّل الخلايا بالترتيب. لا تكتب أن الهدف تحقق إلا إذا أظهرت نتائج تشغيلك ذلك.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. تثبيت المكتبات وتحميل النموذج

In [ ]:
# تثبيت المكتبات المطلوبة (يكفي تشغيله مرة واحدة، ويمكنك وضع # قبله بعد ذلك)
# optimum حُذفت لأنها تتعارض مع إصدار transformers الحديث في Colab؛ نصدّر ONNX بـ torch مباشرة
%pip -q install transformers onnx onnxscript onnxruntime fastapi uvicorn httpx torch numpy

# المكتبات: نكتب فائدة كل مكتبة بجانب اسمها
from pathlib import Path                         # التعامل مع مسارات الملفات
import time                                      # قياس زمن التنفيذ
import re                                        # تنظيف النصوص
import shutil                                    # العثور على أمر تصدير ONNX
import subprocess                                # تشغيل أمر التصدير
import threading                                 # تشغيل API في الخلفية
import numpy as np                               # حساب p50 و p99 ومقارنة النتائج
import torch                                     # تشغيل نموذج PyTorch الأصلي
import onnxruntime as ort                        # تشغيل نموذج ONNX
import httpx                                     # إرسال طلبات HTTP إلى API
import uvicorn                                   # تشغيل خادم FastAPI
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # تحميل tokenizer ونموذج التصنيف
from onnxruntime.quantization import quantize_dynamic, QuantType             # ضغط نموذج ONNX إلى INT8
from fastapi import FastAPI                      # إنشاء API
from pydantic import BaseModel, Field            # التحقق من مدخلات API

# مجلد المشروع على جهازك (غيّره إذا نقلت الملفات)
BASE_DIR = Path("/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data")
print('مجلد العمل | working folder:', BASE_DIR.resolve())
assert BASE_DIR.exists(), f'المجلد غير موجود: {BASE_DIR}'

# البحث عن النموذج وملف الاختبار داخل المجلد وفروعه
def find_first(name, folder_only=False):
    for p in BASE_DIR.rglob(name):
        if p.is_dir() or not folder_only:
            return p
    return None

MODEL_DIR = find_first('bayan_topic_model', folder_only=True)
BENCH_FILE = find_first('bench_mix.npy')
inside = sorted(p.name for p in BASE_DIR.iterdir())   # محتويات المجلد للمساعدة عند الخطأ

assert MODEL_DIR is not None, f'مجلد bayan_topic_model غير موجود. محتويات المجلد: {inside}'
assert (MODEL_DIR / 'config.json').exists(), f'المجلد {MODEL_DIR} لا يحتوي config.json (ليس نموذج transformers)'
assert BENCH_FILE is not None, f'ملف bench_mix.npy غير موجود. محتويات المجلد: {inside}'
DATA_DIR = BENCH_FILE.parent
print('النموذج | model :', MODEL_DIR)
print('الاختبار | bench:', BENCH_FILE)

# تحميل رسائل الاختبار
texts = np.load(BENCH_FILE, allow_pickle=True).tolist()

# تحميل النموذج الأصلي
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
torch.set_num_threads(4)  # استخدام 4 أنوية CPU

print('عدد رسائل الاختبار:', len(texts))
print('تم تحميل النموذج ✅')

مجلد العمل | working folder: /content/drive/.shortcut-targets-by-id/1-vfyPhMTdOHfuv5cox1F4IPardyh6B0c/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data
النموذج | model : /content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/bayan_topic_model
الاختبار | bench: /content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/bench_mix.npy


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

عدد رسائل الاختبار: 2000
تم تحميل النموذج ✅


## 2. قياس سرعة النموذج الأصلي

سنقارن إعدادين:
- `FP32 512`: طول 512 مع Padding ثابت.
- `FP32 128`: طول 128 مع Padding حسب طول النص.

**p50** هو الزمن الوسيط، و**p99** يوضح زمن أبطأ 1% من الطلبات.

In [ ]:
def predict(text, max_length, fixed_padding):
    # تحويل النص إلى أرقام ثم تشغيل النموذج
    inputs = tokenizer(
        str(text),
        max_length=max_length,
        truncation=True,
        padding='max_length' if fixed_padding else True,
        return_tensors='pt'
    )
    with torch.inference_mode():                 # لا نحتاج حسابات التدريب
        output = model(**inputs)
    return int(output.logits.argmax(-1).item())  # إرجاع رقم الفئة المتوقعة


def benchmark(function):
    times = []
    for i in range(100):                         # نكرر القياس 100 مرة
        start = time.perf_counter()
        function(texts[i % len(texts)])
        times.append((time.perf_counter() - start) * 1000)
    return {
        'p50_ms': round(float(np.percentile(times, 50)), 2),
        'p99_ms': round(float(np.percentile(times, 99)), 2)
    }


# قياس الإعدادين
fp32_512 = benchmark(lambda x: predict(x, 512, True))
fp32_128 = benchmark(lambda x: predict(x, 128, False))

print('FP32 512:', fp32_512)
print('FP32 128:', fp32_128)

FP32 512: {'p50_ms': 857.08, 'p99_ms': 2123.94}
FP32 128: {'p50_ms': 148.3, 'p99_ms': 585.82}


## 3. تصدير النموذج إلى ONNX ثم INT8

In [ ]:
# إنشاء مجلد محلي لحفظ نموذج ONNX لتفادي مشكلة نظام الملفات للقراءة فقط
onnx_dir = Path('/content/topic_onnx')
onnx_dir.mkdir(exist_ok=True)

# تصدير النموذج إلى ONNX باستخدام torch (بدون Optimum)
class LogitsOnly(torch.nn.Module):               # غلاف بسيط يعيد درجات الفئات (logits) فقط
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, input_ids, attention_mask):
        return self.base(input_ids=input_ids, attention_mask=attention_mask).logits

export_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation='eager').eval()
dummy = tokenizer(['الخدمة لا تعمل'], max_length=128, truncation=True,
                  padding='max_length', return_tensors='pt')      # مثال واحد يحدد شكل المدخلات
onnx_file = onnx_dir / 'model.onnx'
export_args = dict(
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'sequence'},       # يقبل أي عدد رسائل وأي طول
                  'attention_mask': {0: 'batch', 1: 'sequence'},
                  'logits': {0: 'batch'}},
    opset_version=17
)
with torch.inference_mode():
    try:
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=False, **export_args)      # الطريقة المستقرة
    except Exception as exc:
        print('الطريقة الأولى فشلت، نجرب الثانية:', str(exc)[:200])
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=True, **export_args)       # الطريقة الجديدة (تحتاج onnxscript)
assert onnx_file.exists(), 'لم يُنشأ ملف ONNX'

# إنشاء نسخة INT8 من نموذج ONNX محلياً
int8_file = '/content/topic_int8.onnx'
quantize_dynamic(str(onnx_file), int8_file, weight_type=QuantType.QInt8) #make int8 copy

# فتح النموذجين باستخدام ONNX Runtime
onnx_model = ort.InferenceSession(str(onnx_file), providers=['CPUExecutionProvider'])
int8_model = ort.InferenceSession(int8_file, providers=['CPUExecutionProvider'])
print('تم إنشاء نسختي ONNX و INT8 ✅')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_398/4062550953.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscrip

تم إنشاء نسختي ONNX و INT8 ✅


## 4. قياس السرعة ومقارنة التوقعات

In [ ]:
def predict_onnx(text, session):
    # تجهيز النص بالطريقة نفسها ثم تشغيل نموذج ONNX
    inputs = tokenizer(str(text), max_length=128, truncation=True,
                       padding='max_length', return_tensors='np')
    # نرسل المدخلات التي يحتاجها النموذج فقط
    names = [item.name for item in session.get_inputs()]
    inputs = {name: inputs[name] for name in names}
    output = session.run(None, inputs)[0]
    return int(np.argmax(output, axis=1)[0])


# قياس سرعة ONNX و INT8
onnx_speed = benchmark(lambda x: predict_onnx(x, onnx_model))
int8_speed = benchmark(lambda x: predict_onnx(x, int8_model))
print('ONNX FP32:', onnx_speed)
print('ONNX INT8:', int8_speed)

# مقارنة نتائج INT8 بنتائج النموذج الأصلي على 100 رسالة
sample = texts[:100]
original = [predict(x, 128, False) for x in sample] #original model
compressed = [predict_onnx(x, int8_model) for x in sample] #int8 model
agreement = np.mean(np.array(original) == np.array(compressed)) #إذا كان التوقعان متطابقين نحصل على True (1) حساب نسبة التطابق
print('Prediction agreement:', round(float(agreement), 3))

ONNX FP32: {'p50_ms': 71.94, 'p99_ms': 224.57}
ONNX INT8: {'p50_ms': 75.98, 'p99_ms': 93.27}
Prediction agreement: 0.97


## 5. اختيار النسخة المناسبة

نختار INT8 إذا كانت نتائجه قريبة من النموذج الأصلي. في هذا التمرين نستخدم **97% تطابق** كحد إرشادي، وليس كمعيار عام لكل المشاريع.

نحتفظ دائمًا بنسخة ONNX FP32 كخطة رجوع (Rollback).

In [ ]:
# اختيار النسخة التي ستعمل داخل الخدمة
use_int8 = agreement >= 0.97
selected_model = int8_model if use_int8 else onnx_model

print('النسخة المختارة:', 'INT8' if use_int8 else 'ONNX FP32')
print('سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.')

النسخة المختارة: INT8
سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.


## 6. تجهيز التصنيف وفحوصات البداية (Canary)

الـ Canary اختبار صغير نتأكد به أن تنظيف النص وTokenizer يعملان قبل استقبال المستخدمين.

In [ ]:
# ✅ حل TODO 1: نفس آلية التنظيف من Lab 1 (الدوال الثلاث mask_pii و normalize و preprocess)
import unicodedata

DIGIT_MAP = str.maketrans('٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹', '01234567890123456789')   # الأرقام الهندية -> أرقام عادية
EMAIL_RE = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')        # بريد إلكتروني
PHONE_RE = re.compile(r'(?<!\d)(?:(?:\+|00)966[\s-]?0?|0)5(?:[\s-]?\d){8}(?!\d)')  # جوال سعودي
NID_RE = re.compile(r'(?<!\d)[12]\d{9}(?!\d)')                                  # هوية / إقامة
INVISIBLE_RE = re.compile(r'[​-‏‪-‮﻿]')                # رموز خفية


def normalize(text):
    # NFC + إزالة التطويل والرموز الخفية + توحيد المسافات (لا نحذف الإيموجي ولا علامات الترقيم)
    text = unicodedata.normalize('NFC', str(text)).replace('ـ', '')
    text = INVISIBLE_RE.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()


def mask_pii(text):
    # إخفاء البيانات الشخصية: البريد ثم الجوال ثم الهوية
    text = str(text).translate(DIGIT_MAP)
    text = EMAIL_RE.sub('[EMAIL]', text)
    text = PHONE_RE.sub('[PHONE]', text)
    return NID_RE.sub('[NATIONAL_ID]', text)


def preprocess(text):
    return mask_pii(normalize(text))


def clean_text(text):
    return preprocess(text)    # نفس تنظيف التدريب في Lab 1


def classify(text):
    clean = clean_text(text)
    pred_id = predict_onnx(clean, selected_model)
    label = model.config.id2label.get(pred_id, str(pred_id))
    return {'text': clean, 'topic': label, 'model_version': 'bayan-topic-v1'}


# اختبارات بداية بسيطة
assert clean_text('  testـ  text  ') == 'test text'
assert len(tokenizer('الخدمة لا تعمل في الرياض')['input_ids']) > 2
print('Canary checks: GREEN ✅')

# فحص TODO 1: هل أخفت الدالة رقم الهاتف؟
if '[PHONE]' in clean_text('اتصلوا بي على 0551234567'):
    print('TODO 1 ✅ التنظيف مطابق لـ Lab 1')
else:
    print('TODO 1 ⏳ لم تُكمل بعد: رقم الهاتف لا يزال ظاهرًا في النص')
print(classify('تم خصم المبلغ مرتين من الفاتورة'))

Canary checks: GREEN ✅
TODO 1 ✅ التنظيف مطابق لـ Lab 1
{'text': 'تم خصم المبلغ مرتين من الفاتورة', 'topic': 'billing', 'model_version': 'bayan-topic-v1'}


## 7. إنشاء API

Use FastAPI to build one integrated API that connects the Classification, NER, and Semantic Search components to process complaints and return their results.

## 7. إنشاء API وربط مكونات Bayan

سننشئ **واجهة واحدة** بـ **FastAPI**: تكتب شكوى، فتظهر لك 3 نتائج:
1. 📌 **Classification:** موضوع الشكوى (النموذج الذي دربته في Lab 3a).
2. 🏷️ **NER:** الأسماء والأماكن داخل الشكوى (نموذج Lab 3b) ← **هذا TODO لك**.
3. 🔎 **Semantic Search:** حالات سابقة مشابهة (فهرس FAISS من اليوم الثالث) ← **مكتوب لك كمثال**.

**الأقسام التي ستعدّلها (TODO):**
- ✍️ **TODO 1:** غيّر إعدادات الصفحة (العنوان، نص الزر، لونه) وعدد الحالات المشابهة.
- ✍️ **TODO 2:** فعّل NER بنفس طريقة البحث الدلالي (سطر واحد).
- ✍️ **TODO 3:** جرّب قيمة أخرى للباراميتر `aggregation_strategy`.

> بعد أي تعديل: شغّل خلية **الإيقاف** في آخر اللاب، ثم أعد تشغيل خلية API وخلية التشغيل.

ملاحظة مهمة – حرية تطوير الواجهة (UI Customization):

لست مقيدًا بالتصميم أو الواجهة المقدمة في هذا اللاب. يمكنك تخصيص واجهة المستخدم (UI) بالكامل حسب أفكارك، وتغيير الألوان والتنسيق وطريقة عرض النتائج، أو استخدام مكتبات وأدوات أخرى مثل Streamlit أو Gradio بدلًا من الواجهة الحالية، إذا رغبت بذلك.

كن مبدعًا! الهدف هو بناء تطبيق يعمل ويربط مكونات Bayan بشكل صحيح، وليس الالتزام بتصميم محدد.

In [10]:
# 7.1 تحميل مكونات Bayan (شغّلها كما هي)
%pip -q install sentence-transformers faiss-cpu

import json
import html
import pandas as pd
import faiss                                         # البحث السريع عن أقرب الحالات
from sentence_transformers import SentenceTransformer   # تحويل النص إلى متجه (Embedding)
from transformers import pipeline                    # أدوات جاهزة للنماذج (نستخدمها في NER)

# نبحث عن الملفات داخل مجلد المشروع (الدالة find_first موجودة في الخطوة 1)
CASES_FILE = find_first('bayan_cases.csv')            # الحالات السابقة
INDEX_FILE = find_first('bayan_cases.faiss')          # فهرس البحث من اليوم الثالث
MANIFEST_FILE = find_first('bayan_search_manifest.json')   # اسم نموذج الـ Embedding
NER_DIR = find_first('bayan_ner_model', folder_only=True)  # نموذج NER من Lab 3b

# إذا لم يعثر على ملف الفهرس في الدرايف، نتحقق مما إذا كان قد تم حفظه محلياً سابقاً
if INDEX_FILE is None:
    local_index_path = Path('/content/bayan_cases.faiss')
    if local_index_path.exists():
        INDEX_FILE = local_index_path

search_ready = CASES_FILE is not None
if search_ready:
    cases = pd.read_csv(CASES_FILE)
    EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    if MANIFEST_FILE is not None:                     # نستخدم النموذج نفسه الذي بنى الفهرس
        EMBED_MODEL = json.load(open(MANIFEST_FILE, encoding='utf-8'))['model']
    embedder = SentenceTransformer(EMBED_MODEL)

    if INDEX_FILE is not None:
        faiss_index = faiss.read_index(str(INDEX_FILE))
    else:
        # لا يوجد فهرس: نبنيه الآن محلياً في مسار قابل للكتابة ونحفظه
        print('لا يوجد bayan_cases.faiss، سنبنيه الآن...')
        case_texts = (cases['title'].fillna('').astype(str) + ' ' + cases['description'].fillna('').astype(str)
                      + ' ' + cases['resolution'].fillna('').astype(str)).str.strip().tolist()
        vectors = embedder.encode(case_texts, batch_size=64, show_progress_bar=True,
                                  normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        faiss_index = faiss.IndexFlatIP(vectors.shape[1])
        faiss_index.add(vectors)
        # الحفظ في مسار محلي قابل للكتابة تفادياً لخطأ Read-only file system
        local_save_path = '/content/bayan_cases.faiss'
        faiss.write_index(faiss_index, local_save_path)
        INDEX_FILE = Path(local_save_path)
        print(f'تم حفظ الفهرس بنجاح في: {local_save_path}')

    assert faiss_index.ntotal == len(cases), 'الفهرس لا يطابق ملف الحالات: أعد بناءه من اليوم الثالث'
    print('Semantic Search ✅ عدد الحالات:', len(cases))
else:
    print('Semantic Search ⚠️ ملف bayan_cases.csv غير موجود في المجلد')

print('NER model:', NER_DIR if NER_DIR is not None else '⚠️ مجلد bayan_ner_model غير موجود (أكمل Lab 3b)')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

لا يوجد bayan_cases.faiss، سنبنيه الآن...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

تم حفظ الفهرس بنجاح في: /content/bayan_cases.faiss
Semantic Search ✅ عدد الحالات: 20000
NER model: /content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/bayan_ner_model


In [13]:
from fastapi.responses import HTMLResponse           # لإرجاع صفحة ويب بدل JSON

# ✍️ TODO 1: غيّر هذه القيم ثم أعد تشغيل الخلية (جرّب لونًا آخر مثل '#1f4e9c')
PAGE_TITLE = 'مرحبا بك في بيان لتحليل الشكاوي '      # عنوان الصفحة
BUTTON_TEXT = 'حلّل الشكوى'              # النص على الزر
BUTTON_COLOR = '#0d2745'                  # لون الزر والعنوان
TOP_K = 5                                # عدد الحالات المشابهة التي تظهر
MIN_SCORE = 0.50                           # أقل درجة تشابه (جرّب 0.5 لإخفاء الحالات الضعيفة)


# إنشاء تطبيق FastAPI
app = FastAPI(title="Bayan API")


# شكل البيانات التي سيرسلها المستخدم
class ComplaintRequest(BaseModel):
    text: str = Field(..., min_length=1, description="Arabic or English complaint")


# المكوّن الأول: Classification (جاهز من الخطوات السابقة)
def run_classification(text):
    return classify(text)


# المكوّن الثاني: NER
# ✍️ TODO 2: فعّل NER بنفس فكرة البحث الدلالي (حمّل النموذج مرة واحدة، ثم استخدمه داخل الدالة)
# استبدل  None  بالسطر التالي (انسخه بدون علامة #):
# pipeline('token-classification', model=str(NER_DIR), aggregation_strategy='simple')

#
# ✍️ TODO 3: aggregation_strategy يحدد كيف تُجمع أجزاء الكلمة في كيان واحد.
# جرّب 'simple' ثم 'first' ثم 'max' وقارن النتائج.
ner_pipe = pipeline(
    'token-classification',
    model=str(NER_DIR),
    aggregation_strategy='first'
)


def run_ner(text):
    if ner_pipe is None:
        return []                                     # NER غير مفعّل بعد (TODO 2)
    found = ner_pipe(text)                            # كل عنصر: الكلمة + النوع + الثقة
    return [{'text': e['word'], 'type': e['entity_group'], 'score': round(float(e['score']), 3)}
            for e in found]


# المكوّن الثالث: Semantic Search (مثال جاهز: اقرأه وقارنه بـ NER)
def run_semantic_search(text):
    if not search_ready:
        return []
    # 1) نحوّل الشكوى إلى متجه بالنموذج نفسه الذي بنى الفهرس
    query_vector = embedder.encode([text], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    # 2) نطلب من FAISS أقرب TOP_K حالات
    scores, positions = faiss_index.search(query_vector, TOP_K)
    # 3) نعرض الحالة ودرجة التشابه
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos >= 0 and score >= MIN_SCORE:
            row = cases.iloc[int(pos)]
            results.append({'case_id': str(row['case_id']), 'similarity': round(float(score), 3),
                            'title': str(row['title']), 'description': str(row['description'])[:160]})
    return results


# فحص أن الخدمة تعمل
@app.get("/health")
def health():
    return {"status": "ok"}


# استقبال شكوى وتشغيل المكونات الثلاثة
@app.post("/v1/analyze")
def analyze_complaint(request: ComplaintRequest):
    text = clean_text(request.text)                   # نفس تنظيف Lab 1 (يخفي الهاتف والبريد)
    return {
        "input_text": text,
        "classification": run_classification(text),
        "entities": run_ner(text),
        "similar_cases": run_semantic_search(text)
    }


# الصفحة التي يستخدمها المستخدم (HTML) — لا تحتاج تعديلها، غيّر الإعدادات في الأعلى فقط
PAGE_HTML = """<!doctype html>
<html lang="ar" dir="rtl"><head><meta charset="utf-8"><title>__TITLE__</title>
<style>
  body { font-family: Arial, sans-serif; max-width: 760px; margin: 30px auto; padding: 0 16px; background: #f6f8f7; color: #222; }
  h1 { color: __COLOR__; }
  textarea { width: 100%; height: 110px; font-size: 16px; padding: 10px; box-sizing: border-box; }
  button { background: __COLOR__; color: #fff; border: 0; border-radius: 8px; padding: 12px 22px; font-size: 17px; cursor: pointer; margin-top: 10px; }
  .card { background: #fff; border-radius: 10px; padding: 14px 18px; margin-top: 14px; box-shadow: 0 1px 4px #0002; }
  .tag { display: inline-block; background: #e3f1ea; border-radius: 6px; padding: 3px 9px; margin: 3px; }
</style></head><body>
<h1>__TITLE__</h1>
<textarea id="t" placeholder="اكتب الشكوى هنا بالعربية أو الإنجليزية..."></textarea><br>
<button onclick="go()">__BUTTON__</button>
<div id="out"></div>
<script>
function esc(s) { return String(s).replace(/[&<>"]/g, function (c) { return {'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]; }); }
async function go() {
  var out = document.getElementById('out');
  out.innerHTML = '<div class="card">... جاري التحليل</div>';
  var r = await fetch('/v1/analyze', {method: 'POST', headers: {'Content-Type': 'application/json'},
                                      body: JSON.stringify({text: document.getElementById('t').value})});
  if (!r.ok) { out.innerHTML = '<div class="card">⚠️ اكتب شكوى أولًا</div>'; return; }
  var d = await r.json();
  var h = '<div class="card"><b>📌 الموضوع (Classification):</b> <span class="tag">' + esc(d.classification.topic) + '</span></div>';
  h += '<div class="card"><b>🏷️ الكيانات (NER):</b><br>';
  h += d.entities.length ? d.entities.map(function (e) { return '<span class="tag">' + esc(e.text) + ' ← ' + esc(e.type) + '</span>'; }).join('') : 'لا توجد كيانات';
  h += '</div><div class="card"><b>🔎 حالات مشابهة (Semantic Search):</b>';
  h += d.similar_cases.length ? d.similar_cases.map(function (c) { return '<p>' + esc(c.case_id) + ' (' + c.similarity + ') — ' + esc(c.description) + '</p>'; }).join('') : '<br>لا توجد حالات مشابهة';
  out.innerHTML = h + '</div>';
}
</script></body></html>"""


@app.get("/", response_class=HTMLResponse)
def home():
    return (PAGE_HTML.replace('__TITLE__', html.escape(PAGE_TITLE))
                     .replace('__BUTTON__', html.escape(BUTTON_TEXT))
                     .replace('__COLOR__', html.escape(BUTTON_COLOR)))


print("Bayan API is ready ✅")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Bayan API is ready ✅


### تشغيل API وتجربتها

شغّل الخادم في الخلفية، ثم افتح الرابط الذي سيظهر لك في المتصفح واكتب شكوى.

> إذا أعدت تشغيل هذه الخلية فشغّل خلية **الإيقاف** (آخر خلية) أولًا، وإلا سيكون المنفذ 8000 مشغولًا.

In [14]:
# تشغيل خادم API محليًا
server = uvicorn.Server(
    uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
)

# تشغيل الخادم في Thread منفصل حتى نستطيع إرسال طلبات إليه
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
time.sleep(3)  # انتظار قصير حتى يبدأ الخادم

# التأكد من أن الخادم يعمل
health_response = httpx.get("http://127.0.0.1:8000/health", timeout=10)
print("Health:", health_response.json())

# تجربة إرسال شكوى إلى API
test_response = httpx.post(
    "http://127.0.0.1:8000/v1/analyze",
    json={"text": "تأخر الإصلاح في حي النرجس بالرياض وتم خصم المبلغ مرتين"},
    timeout=60
)
print("Status code:", test_response.status_code)
result = test_response.json()
print("Classification :", result["classification"]["topic"])
print("Entities       :", result["entities"])
print("Similar cases  :", len(result["similar_cases"]))

# فحص الـ TODO
print('TODO 2 ✅ NER يعمل' if result["entities"] else 'TODO 2 ⏳ الكيانات فارغة: نفّذ TODO 2 (أو لا توجد كيانات في هذا النص)')

# الواجهة
print("🌐 افتح الواجهة في المتصفح: http://127.0.0.1:8000")
try:
    from google.colab import output                    # في Colab تظهر الواجهة هنا مباشرة
    output.serve_kernel_port_as_iframe(8000, height=650)
except Exception:
    pass

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


Health: {'status': 'ok'}
Status code: 200
Classification : outage
Entities       : [{'text': 'الإصلاح', 'type': 'SERVICE', 'score': 0.981}, {'text': 'حي', 'type': 'LOCATION', 'score': 0.925}, {'text': 'النرجس', 'type': 'LOCATION', 'score': 0.992}, {'text': 'بالرياض', 'type': 'LOCATION', 'score': 0.995}, {'text': 'خصمالمبلغ', 'type': 'SERVICE', 'score': 0.937}]
Similar cases  : 5
TODO 2 ✅ NER يعمل
🌐 افتح الواجهة في المتصفح: http://127.0.0.1:8000


/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:445: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


<IPython.core.display.Javascript object>

### جرّب التعديل بنفسك (اختياري)

- غيّر عنوان الصفحة ولون الزر ونص الزر (TODO 1) وأعد التشغيل.
- غيّر `TOP_K` إلى 5 و`MIN_SCORE` إلى 0.5 وشاهد تغيّر الحالات المشابهة.
- جرّب `aggregation_strategy='first'` في NER (TODO 3).
- أضف مسارًا جديدًا مثل `/version` (ابحث في توثيق FastAPI: fastapi.tiangolo.com ← Tutorial ← First Steps).
- غيّر اسم المسار `/v1/analyze` (غيّره أيضًا في كود الصفحة `fetch(...)`).

**مهم:** ظهور `[]` في الكيانات يعني أن NER غير مفعّل بعد؛ نفّذ TODO 2.

In [15]:
# إيقاف الخادم بعد الانتهاء من التجربة
server.should_exit = True
api_thread.join(timeout=10)
print("Bayan API stopped.")


Bayan API stopped.
